In [1]:
displayHTML("""
<div style="background: linear-gradient(135deg, #1B3A4B 0%, #2D5F7B 100%); padding: 40px 30px; border-radius: 16px; color: #FFFFFF; font-family: 'Segoe UI', Arial, sans-serif; margin-bottom: 20px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <h1 style="font-size: 2.4em; margin: 0 0 10px 0; font-weight: 700; color: #FFFFFF;">Lab 07 &mdash; Build a Delta Live Tables Pipeline (SOLUTION)</h1>
  <p style="font-size: 1.15em; margin: 0 0 24px 0; opacity: 0.92; color: #FFFFFF;">Azure Databricks Training &bull; Day 2</p>
  <hr style="border: none; border-top: 1px solid rgba(255,255,255,0.3); margin: 0 0 18px 0;">
  <h3 style="margin: 0 0 12px 0; font-weight: 600; color: #FFFFFF;">This is the SOLUTION notebook &mdash; all blanks are filled in.</h3>
</div>
""")

Lab 07 — Build a Delta Live Tables Pipeline (SOLUTION) 
 Azure Databricks Training • Day 2 
 
 This is the SOLUTION notebook — all blanks are filled in.

---
## Configuration

In [ ]:
import dlt
from pyspark.sql.functions import (
    col, current_timestamp, input_file_name, expr,
    count, sum as spark_sum, avg, round as spark_round,
    year, month, to_date, when, lit
)

CATALOG_NAME = "axa_training"  # Change to your catalog name
SCHEMA_ASSETS = "lab_assets"
VOLUME_NAME = "training_volume"

RAW_PLACEMENTS_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_ASSETS}/{VOLUME_NAME}/raw_data/placements"
RAW_ENTITIES_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_ASSETS}/{VOLUME_NAME}/raw_data/entities"

---
## Bronze Layer

In [ ]:
@dlt.table(
    name="bronze_placements",
    comment="Bronze: Raw placements data ingested from training volume via Auto Loader"
)
def bronze_placements():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(RAW_PLACEMENTS_PATH)
        .withColumn("_ingestion_time", current_timestamp())
        .withColumn("_source_file", input_file_name())
    )

---
## Silver Layer

In [ ]:
@dlt.table(
    name="silver_placements",
    comment="Silver: Cleaned placements with data quality expectations"
)
@dlt.expect_or_fail("valid_placement_id", "placement_id IS NOT NULL")
@dlt.expect_or_drop("positive_market_value", "market_value > 0")
@dlt.expect("has_book_value", "book_value IS NOT NULL")
def silver_placements():
    return (
        dlt.read_stream("bronze_placements")
        .withColumn("valuation_date", to_date(col("valuation_date")))
        .withColumn("valuation_year", year(col("valuation_date")))
        .withColumn("valuation_month", month(col("valuation_date")))
        .withColumn(
            "unrealized_gain_loss",
            col("market_value") - col("book_value")
        )
        .withColumn(
            "gain_loss_pct",
            spark_round(
                (col("market_value") - col("book_value")) / col("book_value") * 100,
                2
            )
        )
        .select(
            "placement_id",
            "axa_entity_id",
            "asset_class",
            "instrument_id",
            "market_value",
            "book_value",
            "unrealized_gain_loss",
            "gain_loss_pct",
            "currency",
            "valuation_date",
            "valuation_year",
            "valuation_month",
            "_ingestion_time",
            "_source_file"
        )
    )

---
## Gold Layer

In [ ]:
@dlt.table(
    name="gold_asset_summary",
    comment="Gold: Asset allocation summary by asset class"
)
def gold_asset_summary():
    return (
        dlt.read("silver_placements")
        .groupBy("asset_class")
        .agg(
            spark_sum("market_value").alias("total_market_value"),
            count("*").alias("position_count"),
            spark_round(avg("market_value"), 2).alias("avg_position_value"),
            spark_round(avg("gain_loss_pct"), 2).alias("avg_gain_loss_pct")
        )
    )

In [ ]:
@dlt.table(
    name="gold_entity_performance",
    comment="Gold: Performance summary by AXA entity"
)
def gold_entity_performance():
    return (
        dlt.read("silver_placements")
        .groupBy("axa_entity_id")
        .agg(
            spark_sum("market_value").alias("total_aum"),
            count("*").alias("position_count"),
            spark_round(avg("gain_loss_pct"), 2).alias("avg_gain_loss_pct"),
            spark_sum("unrealized_gain_loss").alias("total_gain_loss")
        )
    )